# Group 27E, Milestone 4: verify the results
Run each cell in order (Runtime → Run all). Takes about 5 minutes.
Each check prints **PASS** or **FAIL** next to the number in the report.

## 1. Get the code and data

In [ ]:
!git clone -q https://github.com/PhonthakornPhreuksa/ieor4004-group27e.git
%cd ieor4004-group27e
!ls

## 2. Install Gurobi (free size-limited license, no sign-up needed for this model size)

In [ ]:
!pip install -q gurobipy yfinance

## 3. Re-run the whole model (this recreates every file in `results/`)

In [ ]:
!python src/run_milestone4.py 2>&1 | grep -v Restricted

## 4. Compare with the numbers in the report

In [ ]:
import sys, numpy as np, pandas as pd
sys.path.append("src")
from data import load

def check(name, got, expected, tol):
    ok = abs(got - expected) <= tol
    print(f"{'PASS' if ok else 'FAIL'}  {name}: got {got:.4g}, report says {expected}")

D = load()
check("Universe size n", len(D["universe"]), 94, 0)
check("Estimation days T", len(D["est"]), 501, 0)
check("Test days", len(D["test"]), 251, 0)
print("Excluded tickers:", D["excluded"], "(report: ALAB, CRWV, HONA, NBIS, SNDK, SPCX)")

### Table 1: parameter summary

In [ ]:
s = pd.read_csv("results/parameter_summary.csv", index_col=0)
mu_ann = s.mu_daily * 252
check("min annual mu (%)", 100*mu_ann.min(), -49.9, 0.1)
check("median annual mu (%)", 100*mu_ann.median(), 26.6, 0.1)
check("max annual mu (%)", 100*mu_ann.max(), 174.2, 0.1)
check("min daily MAD (%)", 100*s.mad_daily.min(), 0.75, 0.01)
check("median daily MAD (%)", 100*s.mad_daily.median(), 1.40, 0.01)
check("max daily MAD (%)", 100*s.mad_daily.max(), 4.34, 0.01)
check("stocks with negative mu", (s.mu_daily < 0).sum(), 9, 0)
print("Top 5 mu:", mu_ann.sort_values(ascending=False).head(5).index.tolist(), "(report: APP, MSTR, RKLB, PLTR, NVDA)")
print("Lowest 5 MAD:", s.mad_daily.sort_values().head(5).index.tolist(), "(report: LIN, ADP, ROP, CSCO, PEP)")

### Table 2: in-sample results

In [ ]:
b = pd.read_csv("results/baseline_results.csv")
opt = b.set_index("portfolio")
base = opt.loc["No limit", "objective"]
expected = {"K=5": (5, 24.0, 36.7, 0.58), "K=10": (10, 5.8, 33.6, 0.50),
            "K=20": (20, 0.25, 33.5, 0.48), "K=50": (25, 0.0, 32.7, 0.48)}
for p, (h, cost, ret, mad) in expected.items():
    r = opt.loc[p]
    check(f"{p} holdings", r.n_hold, h, 0)
    check(f"{p} cost vs no limit (%)", 100*(base - r.objective)/abs(base), cost, 0.05)
    check(f"{p} expected return (%/yr)", 100*r.exp_return_ann, ret, 0.05)
    check(f"{p} daily MAD (%)", 100*r.mad_daily_in, mad, 0.005)
print("All solves optimal:", bool(b.optimal.dropna().astype(bool).all()))
print("Failed constraint checks:", b.checks_failed.dropna().tolist() or "none")

### Sanity checks: shortcut vs MILP, K=5 holdings, holdings by lambda

In [ ]:
n = pd.read_csv("results/check_naive_vs_milp.csv")
for K, pct in zip([5, 10, 20], [8.2, 4.3, 0.2]):
    check(f"MILP better than shortcut, K={K} (%)", n.loc[n.K == K, "milp_better_by_pct"].item(), pct, 0.05)

w = pd.read_csv("results/baseline_weights.csv", index_col=0)["K=5"]
print("K=5 holdings:", (100*w[w > 0].sort_values(ascending=False)).round(1).to_dict(),
      "(report: AEP 30, WMT 27, ORLY 21, TTWO 13, NVDA 9)")

h = pd.read_csv("results/holdings_by_lambda.csv")
print(h.to_string(index=False))
print("(report: 4 stocks at lambda=0, 14 at 0.25, 25 at 1, 28 at lambda>=5)")

### Table 3: out-of-sample (buy-and-hold)

In [ ]:
expected = {"K=5": (-1.7, 12.8, -0.37, -11.8), "K=10": (3.5, 10.4, 0.02, -10.4),
            "K=20": (2.1, 9.6, -0.14, -7.3), "No limit": (3.1, 9.3, -0.04, -6.5),
            "Equal weight": (45.2, 22.6, 1.60, -12.7), "QQQ": (23.8, 20.0, 0.98, -12.0)}
for p, (tr, vol, sh, dd) in expected.items():
    r = opt.loc[p]
    check(f"{p} total return (%)", 100*r.oos_total_return, tr, 0.05)
    check(f"{p} annual vol (%)", 100*r.oos_ann_vol, vol, 0.05)
    check(f"{p} Sharpe", r.oos_sharpe, sh, 0.005)
    check(f"{p} max drawdown (%)", 100*r.oos_max_drawdown, dd, 0.05)

## 5. Independent check without the model code
Recompute the K=5 portfolio's expected return directly from prices and its weights. Should match 36.7% per year in Table 2.

In [ ]:
px = pd.read_csv("data/prices.csv", index_col=0, parse_dates=True)
rets = px.pct_change().loc["2023-10-02":"2025-09-30"]
w5 = w[w > 0]
port = rets[w5.index] @ w5.values
check("K=5 expected return, recomputed by hand (%/yr)", 100*port.mean()*252, 36.7, 0.05)
check("K=5 daily MAD, recomputed by hand (%)", 100*np.mean(np.abs(port - port.mean())), 0.58, 0.005)
print("Weights sum to:", round(w5.sum(), 6), "| min weight:", round(w5.min(), 4), "| max weight:", round(w5.max(), 4))

## 6. Missing risk-free values (assumption A9)

In [ ]:
rf = pd.read_csv("data/dgs3mo.csv", index_col=0, parse_dates=True).iloc[:, 0]
missing = rf.reindex(D["test"].index)
print("Test days with no DGS3MO value:", missing[missing.isna()].index.strftime("%Y-%m-%d").tolist(),
      "(report: 2 days, Oct 13 and Nov 11, 2025)")